# 📝 [제 12편] 오차역전파 (1) - 연쇄법칙으로 한꺼번에 미분하기

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「오차역전파 (I) 개념 • 동작원리 • 시그모이드 함수 미분」, 「오차역전파 (II) 출력층 오차역전파 공식」 (ML_LEC_23, 24)

---

## 🎯 학습 목표
1. 수치미분의 문제점과 오차역전파의 아이디어(연쇄법칙으로 국소 미분 분리)를 설명할 수 있다.
2. sigmoid 함수의 도함수 sigmoid(z) × (1 − sigmoid(z)) 를 유도할 수 있다.
3. 출력층 가중치와 바이어스의 오차역전파 공식을 유도하고, 수치미분 결과와 같음을 확인할 수 있다.

---

## 🧭 지난 시간 복습
* 제 11편: 784 - 100 - 10 신경망을 수치미분으로 학습시키면 사진 1장에 몇 초, 30,000장이면 **수십 시간** 이 걸렸습니다.
* 제 1편: 합성함수의 미분 = **연쇄법칙(chain rule)**,  df/dx = (df/dt) × (dt/dx)

[⬅ 이전 : 제 11편 MNIST - 손글씨 숫자를 읽는 신경망](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/11_mnist.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 13편 오차역전파 (2) - 은닉층 공식과 MNIST 고속 학습 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/13_backprop_2.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분, sigmoid 함수 (실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
print('준비 완료')

## 📖 개념 다지기 1. 오차역전파의 아이디어

가중치 W<sup>(2)</sup> 가 변하면 최종 오차 E 가 얼마나 변하는지(∂E/∂W<sup>(2)</sup>) 알아야 경사하강법을 할 수 있습니다.

```
 입력 → [W(2)] → A(2) → [W(3)] → A(3) → 오차 E
```

W<sup>(2)</sup> 는 E 에 **직접** 영향을 주지 않고, A<sup>(2)</sup> → A<sup>(3)</sup> → E 를 **거쳐서** 영향을 줍니다. 연쇄법칙으로 쪼개면

> ∂E/∂W<sup>(2)</sup> = ( ∂E/∂A<sup>(3)</sup> ) × ( ∂A<sup>(3)</sup>/∂A<sup>(2)</sup> ) × ( ∂A<sup>(2)</sup>/∂W<sup>(2)</sup> )
> 　　　　　　　 ①　　　　　　 ②　　　　　　 ③

* ①, ②, ③ 처럼 이웃한 것끼리의 작은 미분을 **국소(local) 미분** 이라고 합니다.
* 국소 미분을 **수학 공식** 으로 미리 구해 두고, 출력층(①)부터 거꾸로(②, ③) 곱해 나가며 계산하는 방법이 **오차역전파 (Back Propagation)** 입니다.
* 값을 하나씩 움직여 보는 수치미분과 달리 **행렬 공식으로 한 번에** 계산하므로 아주 빠릅니다.

## 📖 개념 다지기 2. 가장 작은 예 - 뉴런 1개

입력 x 하나, 가중치 w, 바이어스 b, 정답 t 인 뉴런 1개를 생각합니다. 손실함수는 계산을 깔끔하게 하려고 **E = ½ (t − a)²** 을 씁니다.

```
  w ──┐
  x ──┼─→ z = wx + b ─→ a = sigmoid(z) ─→ E = ½(t − a)²
  b ──┘
```

연쇄법칙:  **∂E/∂w = (∂E/∂a) × (∂a/∂z) × (∂z/∂w)**

| 국소 미분 | 공식 | x = 1, w = 0.5, b = 0, t = 1 일 때 |
|:-:|:-:|:-:|
| ∂E/∂a | −(t − a) = a − t | 0.6225 − 1 = ( 　　　 ) |
| ∂a/∂z | a(1 − a)  ← 아래 개념 3 | 0.6225 × 0.3775 = ( 　　　 ) |
| ∂z/∂w | x | ( 　 ) |
| **∂E/∂w** | **(a − t) × a(1 − a) × x** | ( 　　　 ) |

(z = 0.5 × 1 + 0 = 0.5,  a = sigmoid(0.5) ≒ 0.6225)

In [ ]:
x, w, b, t = 1.0, 0.5, 0.0, 1.0
z = w*x + b
a = sigmoid(z)
E = 0.5 * (t - a)**2
print(f'피드포워드 : z = {z},  a = {a:.4f},  E = {E:.4f}')

dE_da = a - t
da_dz = a * (1 - a)
dz_dw = x
print(f'국소 미분  : ∂E/∂a = {dE_da:.4f},  ∂a/∂z = {da_dz:.4f},  ∂z/∂w = {dz_dw}')
print(f'오차역전파 : ∂E/∂w = {dE_da * da_dz * dz_dw:.6f}')

# 수치미분과 비교
W_arr = np.array([w])
f = lambda W : 0.5 * (t - sigmoid(W[0]*x + b))**2
print(f'수치미분   : ∂E/∂w = {numerical_derivative(f, W_arr)[0]:.6f}')

> **[확인 1]** 오차역전파 결과와 수치미분 결과가 ( 같다 / 다르다 ).
> ∂E/∂w 가 음수이므로 경사하강법 w ← w − α × ∂E/∂w 에서 w 는 ( 커진다 / 작아진다 ). 그러면 a 는 정답 1 에 ( 가까워진다 / 멀어진다 ).

## 🎬 애니메이션 1. 앞으로 계산(순전파), 뒤로 미분(역전파)

위쪽 값은 **순전파** 로 왼쪽 → 오른쪽 계산된 값, 아래쪽 빨간 값은 **역전파** 로 오른쪽 → 왼쪽 곱해져 오는 미분값입니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (계산 그래프)
boxes = [('w = 0.5', 0), ('z = wx + b', 2.4), ('a = sigmoid(z)', 4.8), ('E = ½(t − a)²', 7.2)]
fwd = ['', f'z = {z:.4f}', f'a = {a:.4f}', f'E = {E:.4f}']
local = ['∂z/∂w = x = 1', '∂a/∂z = a(1−a)', '∂E/∂a = a − t', '']
local_v = [dz_dw, da_dz, dE_da]
back = [dE_da*da_dz*dz_dw, dE_da*da_dz, dE_da, 1.0]           # 각 상자까지 거꾸로 곱한 값 (∂E/∂w, ∂E/∂z, ∂E/∂a, ∂E/∂E)
names = ['∂E/∂w', '∂E/∂z', '∂E/∂a', '∂E/∂E']
frames = [('f', k) for k in range(4) for _ in range(5)] + [('b', k) for k in [3, 2, 1, 0] for _ in range(7)] + [('b', 0)]*8
fig, ax = plt.subplots(figsize=(12, 4.2))
def draw(i):
    mode, k = frames[i]
    ax.clear(); ax.set_xlim(-1, 9); ax.set_ylim(-2.2, 2.2); ax.axis('off')
    for j, (name, x0) in enumerate(boxes):
        on_f = (mode == 'f' and j <= k) or mode == 'b'
        on_b = mode == 'b' and j >= k
        ax.add_patch(plt.Rectangle((x0 - 0.9, -0.45), 1.8, 0.9, fc='#e8f1ff' if on_f else 'white', ec='k', lw=2))
        ax.text(x0, 0, name, ha='center', va='center', fontsize=12)
        if on_f and fwd[j]: ax.text(x0, 0.75, fwd[j], ha='center', fontsize=12, color='navy')
        if on_b: ax.text(x0, -0.95, f'{names[j]} = {back[j]:.4f}', ha='center', fontsize=12, color='crimson', weight='bold')
        if j < 3:
            ax.annotate('', xy=(boxes[j+1][1] - 0.92, 0.1), xytext=(x0 + 0.92, 0.1), arrowprops=dict(arrowstyle='->', color='navy', lw=2 if on_f else 0.5))
            if mode == 'b' and j + 1 >= k:
                ax.annotate('', xy=(x0 + 0.92, -0.2), xytext=(boxes[j+1][1] - 0.92, -0.2), arrowprops=dict(arrowstyle='->', color='crimson', lw=2))
                ax.text((x0 + boxes[j+1][1]) / 2, -1.55, f'× {local[j]}\n(= {local_v[j]:.4f})', ha='center', fontsize=10, color='gray')
    ax.set_title('① 순전파 : 값을 앞으로 계산' if mode == 'f' else '② 역전파 : 국소 미분을 뒤에서부터 곱해 오기', fontsize=14)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=150)
plt.close(fig)
show_anim(anim)

## 📖 개념 다지기 3. sigmoid 함수의 미분 ⭐

sigmoid(z) = 1 / (1 + e<sup>−z</sup>) = (1 + e<sup>−z</sup>)<sup>−1</sup>

① 연쇄법칙:  d/dz (1 + e<sup>−z</sup>)<sup>−1</sup> = −(1 + e<sup>−z</sup>)<sup>−2</sup> × ( 　　　 ) = e<sup>−z</sup> / (1 + e<sup>−z</sup>)²

② 분모 분리:　= {1 / (1 + e<sup>−z</sup>)} × {e<sup>−z</sup> / (1 + e<sup>−z</sup>)}

③ 분자에 +1 − 1:　e<sup>−z</sup> / (1 + e<sup>−z</sup>) = {(1 + e<sup>−z</sup>) − 1} / (1 + e<sup>−z</sup>) = 1 − ( 　　　　　 )

④ 정리하면

> **sigmoid′(z) = sigmoid(z) × (1 − sigmoid(z))**

💡 이미 순전파에서 계산한 a = sigmoid(z) 를 그대로 써서 **a(1 − a)** 로 미분값을 얻습니다. 새로 계산할 것이 거의 없어서 빠릅니다!

In [ ]:
zs = np.linspace(-6, 6, 200)
num = np.array([numerical_derivative(lambda v: sigmoid(v[0]), np.array([zz]))[0] for zz in zs])
plt.figure(figsize=(8, 4))
plt.plot(zs, sigmoid(zs), color='crimson', lw=2, label='sigmoid(z)')
plt.plot(zs, sigmoid(zs)*(1 - sigmoid(zs)), color='green', lw=3, label='공식 sigmoid(z)(1 − sigmoid(z))')
plt.plot(zs[::8], num[::8], 'ko', ms=4, label='수치미분')
plt.legend(); plt.grid(alpha=0.3); plt.title('sigmoid 의 도함수 : 공식 = 수치미분'); plt.show()
print('sigmoid′(0) =', sigmoid(0)*(1 - sigmoid(0)), '  ← 도함수의 최댓값')

> **[확인 2]** sigmoid 도함수의 최댓값은 z = ( 　 ) 일 때 ( 　　 ) 이다.
> 💡 미분값이 최대 0.25 라서, 층을 여러 번 거꾸로 곱하면 값이 점점 작아집니다 (0.25 × 0.25 × …). 이것이 깊은 신경망에서 sigmoid 대신 ReLU 를 쓰는 이유 중 하나입니다.

## 📖 개념 다지기 4. 출력층 가중치의 오차역전파 공식

입력 2개 - 은닉 2개 - 출력 2개 신경망 (원본 강의와 같은 구조)

* 손실:  E = E₁ + E₂,　E₁ = ½ (t₁ − a<sub>1</sub><sup>(3)</sup>)²,　E₂ = ½ (t₂ − a<sub>2</sub><sup>(3)</sup>)²
* 출력층:  z<sub>1</sub><sup>(3)</sup> = a<sub>1</sub><sup>(2)</sup> w<sub>11</sub><sup>(3)</sup> + a<sub>2</sub><sup>(2)</sup> w<sub>12</sub><sup>(3)</sup> + b<sub>1</sub><sup>(3)</sup>,　a<sub>1</sub><sup>(3)</sup> = sigmoid(z<sub>1</sub><sup>(3)</sup>)

w<sub>11</sub><sup>(3)</sup> 는 E₂ 에는 영향을 주지 않으므로 ∂E₂/∂w<sub>11</sub><sup>(3)</sup> = 0

> ∂E/∂w<sub>11</sub><sup>(3)</sup> = ∂E₁/∂a<sub>1</sub><sup>(3)</sup> × ∂a<sub>1</sub><sup>(3)</sup>/∂z<sub>1</sub><sup>(3)</sup> × ∂z<sub>1</sub><sup>(3)</sup>/∂w<sub>11</sub><sup>(3)</sup>
> 　　　　　 = (a<sub>1</sub><sup>(3)</sup> − t₁) × a<sub>1</sub><sup>(3)</sup>(1 − a<sub>1</sub><sup>(3)</sup>) × ( 　　　 )

앞의 두 부분 (a<sub>1</sub><sup>(3)</sup> − t₁) × a<sub>1</sub><sup>(3)</sup>(1 − a<sub>1</sub><sup>(3)</sup>) 을 **출력층 손실 (loss_3)** 이라고 부릅니다.

### 4개의 가중치를 행렬로 모으면

> **loss_3 = (A3 − T) × A3 × (1 − A3)**　　　(같은 자리끼리 곱)
> **∂E/∂W<sup>(3)</sup> = A2<sup>T</sup> · loss_3**　　　(은닉층 출력의 전치 · 출력층 손실)
> **∂E/∂b<sup>(3)</sup> = loss_3**

(2 × 1) · (1 × 2) = (2 × 2) → W<sup>(3)</sup> 와 같은 형상!

> **[연습 1]** ∂E/∂w<sub>21</sub><sup>(3)</sup> 은 출력 노드 ( 　 ) 의 손실에 은닉 노드 ( 　 ) 의 출력을 곱한 값이다.

In [ ]:
# 2 - 2 - 2 신경망 (값은 임의로 정함)
A1 = np.array([[0.5, 0.8]])                  # 입력 (1 × 2)
T  = np.array([[1.0, 0.0]])                  # 정답 (1 × 2)
np.random.seed(3)
W2 = np.random.rand(2, 2); b2 = np.random.rand(2)
W3 = np.random.rand(2, 2); b3 = np.random.rand(2)

def loss():                                  # 피드포워드 + 손실 E = ½ Σ (t − a)²
    A2 = sigmoid(A1 @ W2 + b2)
    A3 = sigmoid(A2 @ W3 + b3)
    return 0.5 * np.sum((T - A3)**2)

# ① 피드포워드
A2 = sigmoid(A1 @ W2 + b2)
A3 = sigmoid(A2 @ W3 + b3)
print('A2 =', A2, ' A3 =', A3, ' E =', loss())

# ② 출력층 오차역전파 공식
loss_3 = (A3 - T) * A3 * (1 - A3)
dW3_bp = A2.T @ loss_3
db3_bp = loss_3[0]
print('\n[오차역전파]  ∂E/∂W3 =\n', dW3_bp, '\n ∂E/∂b3 =', db3_bp)

# ③ 수치미분으로 검증
f = lambda x : loss()
print('\n[수치미분]    ∂E/∂W3 =\n', numerical_derivative(f, W3), '\n ∂E/∂b3 =', numerical_derivative(f, b3))

> **[확인 3]** 두 방법의 결과가 소수점 아래 ( 　 ) 자리까지 같다.

## 🎬 애니메이션 2. w₁₁⁽³⁾ 의 미분이 거꾸로 전달되는 길

2 - 2 - 2 신경망 그림에서 E₁ → a₁⁽³⁾ → z₁⁽³⁾ → w₁₁⁽³⁾ 로 국소 미분이 차례로 곱해지는 모습입니다.

In [ ]:
#@title 🎬 애니메이션 2 실행 (출력층 역전파 경로)
P = {'x1': (0, 2), 'x2': (0, 0), 'h1': (3, 2), 'h2': (3, 0), 'o1': (6, 2), 'o2': (6, 0), 'E1': (8.4, 2), 'E2': (8.4, 0)}
steps = [
    ('E1', 'o1', f'∂E1/∂a1(3) = a1(3) − t1 = {A3[0,0] - T[0,0]:.4f}'),
    ('o1', 'o1', f'× ∂a1(3)/∂z1(3) = a(1 − a) = {A3[0,0]*(1 - A3[0,0]):.4f}'),
    ('o1', 'h1', f'× ∂z1(3)/∂w11(3) = a1(2) = {A2[0,0]:.4f}'),
    (None, None, f'= ∂E/∂w11(3) = {dW3_bp[0,0]:.5f}'),
]
frames = [k for k in range(len(steps)) for _ in range(8)] + [3]*8
fig, ax = plt.subplots(figsize=(11, 4.6))
def draw(f):
    k = frames[f]
    ax.clear(); ax.set_xlim(-0.8, 9.2); ax.set_ylim(-1.6, 2.8); ax.axis('off')
    for a_ in ['x1', 'x2']:
        for b_ in ['h1', 'h2']:
            ax.plot(*zip(P[a_], P[b_]), color='lightgray', lw=1.5)
    for a_ in ['h1', 'h2']:
        for b_ in ['o1', 'o2']:
            hl = (a_, b_) == ('h1', 'o1') and k >= 2
            ax.plot(*zip(P[a_], P[b_]), color='crimson' if hl else 'lightgray', lw=4 if hl else 1.5)
    ax.plot(*zip(P['o1'], (8.0, 2)), color='crimson' if k >= 0 else 'lightgray', lw=4)
    ax.plot(*zip(P['o2'], (8.0, 0)), color='lightgray', lw=1.5)
    for n_, (cx, cy) in P.items():
        if n_.startswith('E'):
            ax.text(cx, cy, n_.replace('E1', 'E1').replace('E2', 'E2'), fontsize=16, ha='center', va='center', color='crimson' if n_ == 'E1' else 'gray')
            continue
        hl = (n_ == 'o1' and k >= 1) or (n_ == 'h1' and k >= 2)
        ax.add_patch(plt.Circle((cx, cy), 0.4, fc='#ffd6d6' if hl else 'white', ec='k', lw=2, zorder=3))
        ax.text(cx, cy, n_, ha='center', va='center', fontsize=12, zorder=4)
    ax.text(4.5, 2.35, 'w11(3)', fontsize=13, color='crimson' if k >= 2 else 'gray')
    ax.text(-0.6, -1.0, '   '.join(st[2] for st in steps[:k+1][:2]), fontsize=11, color='darkred')
    if k >= 2: ax.text(-0.6, -1.45, '   '.join(st[2] for st in steps[2:k+1]), fontsize=11, color='darkred', weight='bold' if k == 3 else 'normal')
    ax.set_title('출력층 가중치 w11(3) 의 오차역전파', fontsize=14)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=180)
plt.close(fig)
show_anim(anim)

## 💻 실습. 출력층만 오차역전파로 학습시켜 보기

출력층의 W3, b3 만 공식으로 갱신하고 (W2, b2 는 고정), 손실이 줄어드는지 확인합니다. 은닉층 공식은 다음 시간에!

In [ ]:
np.random.seed(3)
W2 = np.random.rand(2, 2); b2 = np.random.rand(2)
W3 = np.random.rand(2, 2); b3 = np.random.rand(2)
lr = 0.5
for step in range(201):
    A2 = sigmoid(A1 @ W2 + b2)
    A3 = sigmoid(A2 @ W3 + b3)
    loss_3 = (A3 - T) * A3 * (1 - A3)
    W3 -= lr * (A2.T @ loss_3)               # ∂E/∂W3 = A2ᵀ · loss_3
    b3 -= lr * loss_3[0]                     # ∂E/∂b3 = loss_3
    if step % 40 == 0:
        print(f'step {step:3d} : E = {loss():.6f},  A3 = {A3.round(4)}  (정답 {T})')

## ✏️ 연습 문제

> **[문제 1]** 뉴런 1개에서 x = 2, w = 0, b = 0, t = 0 일 때 ∂E/∂w 를 손으로 구하세요. (sigmoid(0) = 0.5)
> z = ( 　 ), a = ( 　 ), ∂E/∂a = ( 　 ), ∂a/∂z = ( 　 ), ∂z/∂w = ( 　 ) → ∂E/∂w = ( 　　 )

> **[문제 2]** 바이어스 b 에 대한 미분 ∂E/∂b 는 ∂z/∂b = ( 　 ) 이므로 (a − t) × a(1 − a) × ( 　 ) 이다. 코드로 확인하세요.

In [ ]:
x, w, b, t = 2.0, 0.0, 0.0, 0.0
a = sigmoid(w*x + b)
dE_dw = ______
dE_db = ______
print('∂E/∂w =', dE_dw, ', ∂E/∂b =', dE_db)
print('수치미분 ∂E/∂w =', numerical_derivative(lambda W: 0.5*(t - sigmoid(W[0]*x + b))**2, np.array([w])))

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('문제 1 : z = 0, a = 0.5, ∂E/∂a = 0.5, ∂a/∂z = 0.25, ∂z/∂w = 2 → ∂E/∂w = 0.25')
print('문제 2 : ∂z/∂b = 1 → ∂E/∂b = (a − t) × a(1 − a) × 1 = 0.125')
print('   빈칸 : dE_dw = (a - t) * a * (1 - a) * x,   dE_db = (a - t) * a * (1 - a)')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 오차역전파 | 연쇄법칙으로 국소 미분을 분리하고, 출력층부터 거꾸로 곱해 미분값을 공식으로 계산 |
| sigmoid 미분 | sigmoid′(z) = sigmoid(z)(1 − sigmoid(z)) = a(1 − a) |
| 출력층 손실 | loss_3 = (A3 − T) × A3 × (1 − A3) |
| 출력층 공식 | ∂E/∂W<sup>(3)</sup> = A2<sup>T</sup> · loss_3,  ∂E/∂b<sup>(3)</sup> = loss_3 |
| 검증 | 공식 결과 = 수치미분 결과, 하지만 훨씬 빠르다 |

### 🏆 도전 과제
은닉층 가중치 W<sup>(2)</sup> 는 출력 노드 **두 개 모두** (E₁, E₂) 에 영향을 줍니다. 그렇다면 ∂E/∂w<sub>11</sub><sup>(2)</sup> 를 구할 때는 연쇄법칙의 길이 몇 갈래가 될까요? ( 　 ) 갈래 → 다음 시간: **은닉층 오차역전파와 일반 공식**

---

[⬅ 이전 : 제 11편 MNIST - 손글씨 숫자를 읽는 신경망](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/11_mnist.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/00_index.ipynb)　|　[다음 : 제 13편 오차역전파 (2) - 은닉층 공식과 MNIST 고속 학습 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/main/13_backprop_2.ipynb)